In [51]:
import pandas as pd
from geographiclib.geodesic import Geodesic
import geopandas
import math
import folium
from shapely.geometry import Point
from shapely.geometry import Polygon
import numpy as np
import matplotlib.pyplot as plt
import random as rand
geod = Geodesic.WGS84 

In [52]:
data = 'obc_fishing.csv'
df = pd.read_csv(data)

In [53]:
print(df.dtypes)

MSG_TYPE                 int64
MMSI                     int64
NAME                    object
IMO_NUMBER             float64
CALL_SIGN               object
LAT_AVG                float64
LON_AVG                float64
PERIOD                  object
SPEED_KNOTS            float64
COG_DEG                float64
HEADING_DEG            float64
NAV_STATUS             float64
NAV_SENSOR             float64
SHIP_AND_CARGO_TYPE      int64
DRAUGHT                float64
DRAUGHT.1              float64
DIM_BOW                  int64
DIM_STERN                int64
DIM_PORT                 int64
DIM_STARBOARD            int64
DESTINATION             object
MMSI_COUNTRY_CD         object
RECEIVER                object
dtype: object


In [54]:
df['PERIOD'] = pd.to_datetime(df['PERIOD'])
print(df['PERIOD'].dtypes)

datetime64[ns]


In [55]:
df = df[['MMSI', 'NAME', 'LAT_AVG', 'LON_AVG', 'PERIOD', 'SPEED_KNOTS', 'COG_DEG', 'HEADING_DEG', 'NAV_STATUS', 'SHIP_AND_CARGO_TYPE', 'DRAUGHT', 'DIM_BOW', 'DIM_STERN', 'DIM_PORT', 'DIM_STARBOARD']]
df['BEAM'] = df['DIM_PORT'] + df['DIM_STARBOARD']
df['LENGTH'] = df['DIM_BOW'] + df['DIM_STERN']

In [56]:
df['channel_side'] = ['Northwestbound' if x > 200 else 'Southeastbound' for x in df['COG_DEG']]
df = df.sort_values(['MMSI', 'PERIOD']).reset_index(drop=True)
df['time_diff'] = (df.groupby('MMSI')['PERIOD'].diff().dt.total_seconds())
#df['cog_diff'] = (df.groupby('MMSI')['COG_DEG'].diff())
#df['cog_diff'] = np.abs((df['cog_diff'] + 180) % 360 -180)
df

,MMSI,NAME,LAT_AVG,LON_AVG,PERIOD,SPEED_KNOTS,COG_DEG,HEADING_DEG,NAV_STATUS,SHIP_AND_CARGO_TYPE,DRAUGHT,DIM_BOW,DIM_STERN,DIM_PORT,DIM_STARBOARD,BEAM,LENGTH,channel_side,time_diff
0,316016142,,22.684000,-78.199567,2023-04-02 16:10:00,5.7,311.2,NaN,NaN,30,NaN,0,0,0,0,0,0,Northwestbound,NaN
1,316016142,,22.687631,-78.204614,2023-04-02 16:15:00,6.7,306.0,NaN,NaN,30,NaN,0,0,0,0,0,0,Northwestbound,300.0
2,316016142,,22.693175,-78.212412,2023-04-02 16:20:00,6.5,303.9,NaN,NaN,30,NaN,0,0,0,0,0,0,Northwestbound,300.0
3,316016142,,22.697255,-78.218239,2023-04-02 16:25:00,6.0,309.2,NaN,NaN,30,NaN,0,0,0,0,0,0,Northwestbound,300.0
4,316016142,,22.702390,-78.225362,2023-04-02 16:30:00,5.5,311.1,NaN,NaN,30,NaN,0,0,0,0,0,0,Northwestbound,300.0
5,316016142,,22.707520,-78.232470,2023-04-02 16:35:00,6.0,304.5,NaN,NaN,30,NaN,0,0,0,0,0,0,Northwestbound,300.0
6,316016142,,22.712671,-78.239580,2023-04-02 16:40:00,5.8,308.9,NaN,NaN,30,NaN,0,0,0,0,0,0,Northwestbound,300.0
7,316016142,,22.717940,-78.246712,2023-04-02 16:45:00,5.8,310.0,NaN,NaN,30,NaN,0,0,0,0,0,0,Northwestbound,300.0
8,316016142,,22.723165,-78.253655,2023-04-02 16:50:00,5.9,307.7,NaN,NaN,30,NaN,0,0,0,0,0,0,Northwestbound,300.0
9,316016142,,22.728571,-78.260574,2023-04-02 16:55:00,6.2,312.9,NaN,NaN,30,NaN,0,0,0,0,0,0,Northwestbound,300.0


In [57]:
print(df['MMSI'].nunique())

7


In [58]:
data = pd.read_csv('obc_boundaries.csv')
data['geometry'] = data.apply(lambda x: (float(x.Long), float(x.Lat)), axis=1)
channel_coords = list(data['geometry'])
channel_coords_north = [channel_coords[0], channel_coords[1], channel_coords[2], channel_coords[3], channel_coords[-1], channel_coords[-2], channel_coords[-3], channel_coords[-4]]
channel_coords_south = [channel_coords[4], channel_coords[5], channel_coords[6], channel_coords[7], channel_coords[-1], channel_coords[-2], channel_coords[-3], channel_coords[-4]]
coords_north = [channel_coords[0], channel_coords[1], channel_coords[2], (-77.44235, 22.18378333), (-77.44235, 23.80836667), (-78.72296667, 23.80836667)]
coords_south = [channel_coords[4], channel_coords[5], channel_coords[6], channel_coords[7], (-77.4681,21.15348333), (-78.75143333, 21.15348333)]
channel_polygon_n = Polygon(channel_coords_north)
channel_polygon_s = Polygon(channel_coords_south)
polygon_n = Polygon(coords_north)
polygon_s = Polygon(coords_south)

In [59]:
def where_is_vessel(LAT_AVG, LON_AVG, BOOL_northc, BOOL_southc, BOOL_north, BOOL_south):
    if BOOL_northc == True:
        answer = 'in north channel'
    elif BOOL_southc == True:
        answer = 'in south channel'
    elif LON_AVG < -78.72296667:
        answer = 'west'
    elif LON_AVG > -77.49385:
        answer = 'east'
    elif BOOL_north == True:
        answer = 'north'
    elif BOOL_south == True:
        answer = 'south'
    else:
        answer = 'other'
    return answer

In [60]:
geo_df = geopandas.GeoDataFrame(df, geometry=geopandas.points_from_xy(df['LON_AVG'], df['LAT_AVG']), crs='EPSG:4326')
geo_df['in_channel_north'] = geo_df.within(channel_polygon_n)
geo_df['in_channel_south'] = geo_df.within(channel_polygon_s)
geo_df['north_of_channel'] = geo_df.within(polygon_n)
geo_df['south_of_channel'] = geo_df.within(polygon_s)
geo_df['location'] = geo_df.apply(lambda x: where_is_vessel(x.LAT_AVG, x.LON_AVG, x.in_channel_north, x.in_channel_south, x.north_of_channel, x.south_of_channel), axis=1)
geo_df.shape

(39, 25)

In [61]:
geo_df['location'].value_counts()

location
north               16
in south channel    14
west                 4
east                 4
south                1
Name: count, dtype: int64

In [62]:
data = pd.read_csv('obc_boundaries.csv')
from shapely.geometry import Point
from shapely.geometry import Polygon
data = data[data['Line']!='Middle']
data['geometry'] = data.apply(lambda x: (float(x.Long), float(x.Lat)), axis=1)

channel_coords = list(data['geometry'])
channel_coords_reorder = [channel_coords[0], channel_coords[1], channel_coords[2], channel_coords[3], channel_coords[-1], channel_coords[-2], channel_coords[-3], channel_coords[-4]]
channel_polygon = Polygon(channel_coords_reorder)
geo_df_channel = geopandas.GeoDataFrame(geometry=[channel_polygon], crs='EPSG:4326')

In [63]:
my_map = folium.Map(location=[22.5, -77.8], zoom_start=7)
folium.GeoJson(geo_df_channel).add_to(my_map)

def rand_color():
    return "#{:06x}".format(rand.randint(0, 0xFFFFFF))

for id, vessel in df.groupby('MMSI'):
    coords = list(vessel[['LAT_AVG', 'LON_AVG']].itertuples(index=False, name=None))

    folium.PolyLine(
        coords,
        color=rand_color(),
        weight=2,
        opacity=0.8,
        popup=f"MMSI: {id}"
    ).add_to(my_map)

my_map